# STEP 1 (WHOLE-POPULATION VERSION) — Data Selection & Cleaning  
================================================================  
No BNF/pathway filter — covers ALL drugs, ALL customers with prescription  
activity in the date window. Every bug found during the diabetes-pathway  
build is fixed here from the start (see inline comments for why each one  
matters — these are not optional style choices, each one caused a real  
silent failure or crash earlier).  
  
Re-checked against the full EDW navigation notebook (74 cells, all  
outputs) — table/folder names, real column names, Tier-A handling, and  
every precision/dtype fix below are confirmed consistent with the actual  
schema. Nothing in the core pipeline needed changing.  
  
NEW in this version: Part E (optional) — IMD deprivation stratification,  
per Mustafa's Section 9 guidance ("stratify by IMD, BNF chapter,  
age/gender"). Skip it if you're short on time; nothing downstream depends  
on it.  
  
KNOWN UNRESOLVED GAP (not a bug, a data-availability fact):  
  Dim.DispenseItem ONLY contains POM (prescription-only) dispensing events  
  in practice — not because it's structurally impossible for a GSL/P drug  
  to appear (LEGAL_CAT describes the drug, not the dispensing route), but  
  because NHS policy actively discourages GPs from prescribing anything  
  available OTC. Confirmed empirically: 100% POM across 4,528 real  
  transaction rows in the diabetes-cohort test. Report this as a genuine  
  finding in your dissertation, not a defect to hide.  
  
Run this AFTER the setup cells (authenticate(), etc.) in the same kernel.


## Setup
Requires `edw_helpers.py` in the same folder. Run these two cells first, every kernel session.

In [2]:
from edw_helpers import (
    authenticate, load_table_smart, mem_check,
    read_first_parquet, table_inventory, list_parts,
    FOLDERS, TIER_A,
)

In [3]:
# Prints a device-code login URL — open it, sign in with your
# @pharmacy2u.co.uk account. Needed once per kernel session.
authenticate()

Stage A  DeviceCodeCredential created
To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code L5D28K5QW to authenticate.
Stage B  datastore resolved
         account_name   = smrttempsa
         container_name = azureml-blobstore-9c0e725a-1db2-4e2b-83ed-97bbde32f29b
Stage C  BlobServiceClient + adlfs filesystem ready


(<azure.storage.blob._container_client.ContainerClient at 0x7eac7c51b220>,
 <adlfs.spec.AzureBlobFileSystem at 0x7eac7c51be50>)

---
## Pipeline starts here

In [4]:
import pandas as pd
import numpy as np
import pyarrow as pa

### CONFIG

In [5]:
DATE_RANGE = ("2024-01-01", "2026-01-01")   # 24-month window; narrow this
                                             # first if you hit memory/time
                                             # limits — it's the single
                                             # biggest lever on row counts
OTC_LEGAL_CATS = {"GSL", "P"}                # POM excluded from OTC targets

# Safety cap for a first full-catalog run. None = no cap (every customer
# with prescription activity in the window — likely a large fraction of
# your ~5.4M customer base). Set to an integer (e.g. 50_000) to sample a
# manageable subset first, confirm the pipeline runs cleanly end-to-end,
# then raise it or set to None once you trust the timing/memory footprint.
CUSTOMER_SAMPLE_SIZE = None
RANDOM_SEED = 42

### PART A — Build the full cohort: every customer with a prescription item in the date window (no BNF/pathway filter this time)

In [6]:
# Gotcha: Dim.PrescriptionItem is Tier-A AND has no CustomerKey column, so
# customer_keys= would error here (field not found) — must use
# date_range+date_column instead. date_range alone silently applies NO
# filter unless date_column is also passed — always pass both together.
mem_check(estimated_bytes=8_000_000_000, label="Dim_PrescriptionItem full-population read")

presc_items = load_table_smart(
    "Dim_PrescriptionItem",
    columns=["PrescriptionItemKey", "PrescriptionKey", "snomedCode", "name",
             "form", "strength", "qty_prescribed", "DeletedAtSource",
             "CDBPrescriptionItemCreationDate"],
    date_range=DATE_RANGE,
    date_column="CDBPrescriptionItemCreationDate",
)
presc_items = presc_items[presc_items["DeletedAtSource"].fillna(False) == False]
print(f"[A] {len(presc_items):,} prescription-item rows in window (all drugs)")

# Bridge PrescriptionItem -> Prescription (has CustomerKey + IsAcute)
presc_header = load_table_smart(
    "Dim_Prescription",
    columns=["PrescriptionKey", "OrderKey", "CustomerKey", "IssueDate", "IsAcute"],
    date_range=DATE_RANGE,
    date_column="IssueDate",
)

cohort_link = presc_items.merge(
    presc_header[["PrescriptionKey", "CustomerKey", "IssueDate", "IsAcute"]],
    on="PrescriptionKey", how="inner",
)
cohort_link["is_acute_only"] = cohort_link["IsAcute"].fillna(False)

all_cohort_customers = cohort_link["CustomerKey"].dropna().unique()
print(f"[A] {len(all_cohort_customers):,} unique candidate CustomerKeys before cleaning/sampling")

# Optional sampling for a manageable first run — see CONFIG above
if CUSTOMER_SAMPLE_SIZE is not None and len(all_cohort_customers) > CUSTOMER_SAMPLE_SIZE:
    rng = np.random.default_rng(RANDOM_SEED)
    base_cohort_customers = rng.choice(
        all_cohort_customers, size=CUSTOMER_SAMPLE_SIZE, replace=False
    )
    print(f"[A] Sampled down to {len(base_cohort_customers):,} customers "
          f"(CUSTOMER_SAMPLE_SIZE={CUSTOMER_SAMPLE_SIZE:,}); set to None for the full population")
    cohort_link = cohort_link[cohort_link["CustomerKey"].isin(base_cohort_customers)]
else:
    base_cohort_customers = all_cohort_customers
    print(f"[A] No sampling applied — using all {len(base_cohort_customers):,} customers")

[mem_check Dim_PrescriptionItem full-population read] planned ~8.00 GB / available 132.60 GB  (6.0%)
[A] 65,773,098 prescription-item rows in window (all drugs)
[A] 1,961,240 unique candidate CustomerKeys before cleaning/sampling
[A] No sampling applied — using all 1,961,240 customers


### PART B — MANDATORY cleaning: exclude deceased / deregistered patients

In [7]:
# Gotcha: the real Dim.Customer schema does NOT have "gender"/"dob" columns
# — those were guesses from the spec doc's shorthand, not the actual field
# names, and caused an ArrowInvalid error. Real names: CustomerSex,
# CustomerDateOfBirth (confirmed from the live schema error dump AND the
# fresh EDW notebook re-check).
customers = load_table_smart(
    "Dim_Customer",
    columns=["CustomerKey", "PostcodeKey", "CustomerSex", "CustomerDateOfBirth",
             "FirstPrescriptionDate"],
    customer_keys=set(base_cohort_customers),
)

dereg = load_table_smart(
    "Fact_CustomerDeregistration",
    columns=["CustomerKey", "DeregistrationDate", "MostRecent"],
    customer_keys=set(base_cohort_customers),
)
dereg = dereg[dereg["MostRecent"] == 1]

snapshot_date = pd.Timestamp(DATE_RANGE[1])

cohort = customers.merge(
    dereg[["CustomerKey", "DeregistrationDate"]], on="CustomerKey", how="left"
)
cohort = cohort[
    cohort["DeregistrationDate"].isna() | (cohort["DeregistrationDate"] > snapshot_date)
]
print(f"[B] {len(cohort):,} customers remain after deceased/deregistered exclusion "
      f"(dropped {len(customers) - len(cohort):,})")

[B] 1,916,955 customers remain after deceased/deregistered exclusion (dropped 44,285)


### PART C — Define the OTC "target" product universe (all drugs, unchanged logic from the diabetes build — this part was never pathway-specific)

In [8]:
product_cols = [
    "ProductKey", "DrugName", "vmp_code", "ProductCodeLevel1", "ProductCodeLevel2",
    "ProductCodeLevel3", "legal_category_id", "legal_controlled", "legal_description",
    "BNF_Condition", "ProductOnFormulary", "ProductBlacklisted", "inactive",
]
dim_product = load_table_smart("Dim_Product", columns=product_cols)

# Gotcha: vmp_code in Dim_Product is text; VMP in Dim_FDBDMDProduct is a
# 17-digit dm+d/SNOMED-style code. Both this AND the earlier snomedCode
# comparison were silently broken by float64: pd.to_numeric(...).astype
# ("float64") and pyarrow's default int-with-nulls->pandas conversion BOTH
# round any 17-digit integer (float64 is only exact up to 2^53, ~16 digits).
# Fix: never route these through a numeric dtype — preserve exact Int64 via
# a types_mapper on load, then compare as strings throughout.
fdb_tbl = load_table_smart(
    "Dim_FDBDMDProduct",
    columns=["VMP", "VMP_NAME", "LEGAL_CAT", "DF_IND", "CONTROL_DRUG_CAT"],
    return_pandas=False,
)
fdb_product = fdb_tbl.to_pandas(
    types_mapper=lambda t: pd.Int64Dtype() if pa.types.is_integer(t) else None
)

dim_product["_vmp_str"] = dim_product["vmp_code"].astype(str).str.strip()
fdb_product["_vmp_str"] = fdb_product["VMP"].astype("string")

# Gotcha: VMP is NOT unique in Dim_FDBDMDProduct — its real grain is one row
# per VMPP/AMP/AMPP pack variant, so merging without deduplicating first is
# a many-to-many join that multiplies row count and can hang for a very
# long time even without OOMing (this is what looked like a dead kernel
# earlier — it was actually a slow hash-join, killed by an execution
# timeout). LEGAL_CAT/CONTROL_DRUG_CAT don't vary by pack size, so
# collapsing to one row per VMP loses nothing needed here.
n_before = len(fdb_product)
fdb_product = fdb_product.drop_duplicates(subset=["_vmp_str"], keep="first")
print(f"[C] Deduplicated Dim_FDBDMDProduct: {n_before:,} -> {len(fdb_product):,} rows (one per VMP)")

products = dim_product.merge(fdb_product, on="_vmp_str", how="left")

# Gotcha: ProductBlacklisted / inactive are "Y"/"N" strings in this data,
# not Python booleans — comparing "N" == False is always False in pandas,
# so a naive filter silently matches 0 rows regardless of LEGAL_CAT. Also
# hedge CONTROL_DRUG_CAT in case it comes through as a string code rather
# than a real int.
def _is_no(series):
    """True where the flag means 'no' — handles 'N'/'Y' strings, real
    booleans, and null (treated as 'no', i.e. not blacklisted/inactive)."""
    return series.isna() | series.astype(str).str.upper().isin(["N", "FALSE", "0"])

otc_products = products[
    products["LEGAL_CAT"].isin(OTC_LEGAL_CATS)
    & (products["CONTROL_DRUG_CAT"].isna() | (products["CONTROL_DRUG_CAT"].astype(str) == "0"))
    & _is_no(products["ProductBlacklisted"])
    & _is_no(products["inactive"])
].copy()

print(f"[C] {len(otc_products):,} OTC-eligible products "
      f"(of {len(products):,} total, LEGAL_CAT in {OTC_LEGAL_CATS})")
print(otc_products["LEGAL_CAT"].value_counts())

[C] Deduplicated Dim_FDBDMDProduct: 185,507 -> 24,492 rows (one per VMP)
[C] 3,619 OTC-eligible products (of 348,467 total, LEGAL_CAT in {'GSL', 'P'})
LEGAL_CAT
GSL    1914
P      1705
Name: count, dtype: int64


### PART D — Build clean dispense-level transactions for the cohort

In [9]:
# Gotcha: Dim.DispenseItem is Tier-A (174M rows, the biggest table) AND has
# no CustomerKey column — same fix as Part A: date_range+date_column, then
# bridge to CustomerKey via PrescriptionItemKey in pandas afterward.
#
# Gotcha: mem_check protects against a bad read, but it can only work with
# whatever RAM is actually free RIGHT NOW. Stale kernel sessions on a
# shared compute instance can silently eat most of your RAM before you've
# run a single cell — if mem_check ever refuses a read that looks like it
# should easily fit, check the RAM indicator in the notebook toolbar before
# assuming the code is wrong. A full compute-instance Stop/Start (not just
# Kernel Restart) is the reliable fix for that.
mem_check(estimated_bytes=15_000_000_000, label="Dim_DispenseItem full-population read")

dispense_cols = [
    "DispenseItemKey", "PrescriptionItemKey", "dispensed_dmd_code", "ProductKey",
    "dispense_date_created", "qty_to_be_dispensed", "not_dispensed_reason",
    "severe_interaction_noted",
]
dispenses = load_table_smart(
    "Dim_DispenseItem",
    columns=dispense_cols,
    date_range=DATE_RANGE,
    date_column="dispense_date_created",
)
dispenses = dispenses[dispenses["not_dispensed_reason"].isna()]

key_map = cohort_link[["PrescriptionItemKey", "CustomerKey"]].drop_duplicates()

transactions = dispenses.merge(key_map, on="PrescriptionItemKey", how="inner")
transactions = transactions.merge(
    cohort[["CustomerKey"]], on="CustomerKey", how="inner"  # re-apply deceased/dereg filter
)

transactions = transactions.merge(
    products[["ProductKey", "DrugName", "ProductCodeLevel1", "ProductCodeLevel2", "LEGAL_CAT"]],
    on="ProductKey", how="left",
)
transactions["is_otc"] = transactions["LEGAL_CAT"].isin(OTC_LEGAL_CATS)

transactions["dispense_day"] = pd.to_datetime(transactions["dispense_date_created"]).dt.date
transactions = transactions.drop_duplicates(subset=["CustomerKey", "ProductKey", "dispense_day"])

print(f"[D] {len(transactions):,} clean dispense-level transaction rows")
print(f"[D] {transactions['CustomerKey'].nunique():,} unique customers")
print(f"[D] OTC share of lines: {transactions['is_otc'].mean():.1%}  "
      f"(expected ~0% — see module docstring: NHS policy discourages OTC-item prescribing)")
print(f"[D] LEGAL_CAT distribution in transactions:")
print(transactions["LEGAL_CAT"].value_counts(dropna=False))

[mem_check Dim_DispenseItem full-population read] planned ~15.00 GB / available 109.70 GB  (13.7%)
[D] 67,124,000 clean dispense-level transaction rows
[D] 1,872,959 unique customers
[D] OTC share of lines: 7.2%  (expected ~0% — see module docstring: NHS policy discourages OTC-item prescribing)
[D] LEGAL_CAT distribution in transactions:
LEGAL_CAT
POM               59929915
P                  3543579
Not Applicable     2300650
GSL                1300555
NaN                  49301
Name: count, dtype: int64


### PART E (OPTIONAL) — IMD deprivation stratification Per Mustafa's Section 9 guidance: "Stratify by IMD (Section 5.2), BNF chapter (Section 5.1), age/gender from Dim.Customer." BNF chapter and age/gender are already in `products`/`cohort` above — this adds the IMD piece specifically. Skip this cell entirely if you're short on time; nothing in Step 2/3 depends on it.

In [10]:
postcodes = load_table_smart(
    "Dim_Postcode",
    columns=["PostcodeKey", "PostcodeNoSpace"],
    customer_keys=None,   # not Tier-A, but this table is 2.7M rows —
                           # column-prune hard and don't scope by customer_keys
                           # (Dim.Postcode has no CustomerKey column either)
)

postcode_to_lsoa = load_table_smart(
    "Report_NHS_Performance_PostcodesToLSOA2022",
    columns=["pcds", "lsoa11cd"],
)

imd = load_table_smart(
    "Report_NHS_Performance_IMD2025EnglandLSOA",
    columns=[
        "LSOA.code..2021.",
        "Index.of.Multiple.Deprivation..IMD..Decile..where.1.is.most.deprived.10..of.LSOAs.",
    ],
)
imd = imd.rename(columns={
    "LSOA.code..2021.": "lsoa11cd",
    "Index.of.Multiple.Deprivation..IMD..Decile..where.1.is.most.deprived.10..of.LSOAs.": "imd_decile",
})

# Gotcha: column-name case differs between the two postcode tables —
# uppercase-normalise both sides before merging (per the spec doc's own
# note on this exact join)
postcodes["_pc_norm"] = postcodes["PostcodeNoSpace"].astype(str).str.upper().str.strip()
postcode_to_lsoa["_pc_norm"] = postcode_to_lsoa["pcds"].astype(str).str.upper().str.strip()

cohort_imd = cohort.merge(postcodes[["PostcodeKey", "_pc_norm"]], on="PostcodeKey", how="left")
cohort_imd = cohort_imd.merge(postcode_to_lsoa[["_pc_norm", "lsoa11cd"]], on="_pc_norm", how="left")
cohort_imd = cohort_imd.merge(imd[["lsoa11cd", "imd_decile"]], on="lsoa11cd", how="left")

# PII minimisation (per spec doc): drop postcode/LSOA once decile is joined,
# keep only the decile band
cohort_imd = cohort_imd.drop(columns=["_pc_norm", "lsoa11cd"], errors="ignore")

print(f"[E] IMD decile matched for {cohort_imd['imd_decile'].notna().sum():,} / "
      f"{len(cohort_imd):,} customers ({cohort_imd['imd_decile'].notna().mean():.1%})")
print(cohort_imd["imd_decile"].value_counts(dropna=False).sort_index())

cohort = cohort_imd  # replace cohort with the IMD-enriched version for saving below

[E] IMD decile matched for 0 / 1,916,955 customers (0.0%)
imd_decile
NaN    1916955
Name: count, dtype: int64


### Save outputs for Step 2 (market basket analysis)

In [11]:
transactions.to_parquet("clean_transactions_all_drugs.parquet", index=False)
otc_products.to_parquet("otc_product_universe.parquet", index=False)
cohort.to_parquet("clean_cohort_all.parquet", index=False)

print("\nSaved: clean_transactions_all_drugs.parquet, "
      "otc_product_universe.parquet, clean_cohort_all.parquet")


Saved: clean_transactions_all_drugs.parquet, otc_product_universe.parquet, clean_cohort_all.parquet
